# **01 LangChain 기초**

### 학습 내용
1. Chat Models 초기화 및 사용법
2. 모델 invocation 방식 (invoke, stream, batch)
3. Messages의 구조와 타입 이해
4. 대화 기록 관리
5. 토큰 사용량 추적

## 1. 환경 변수 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("API Key가 설정되었습니다.")

API Key가 설정되었습니다.


**LangChain**은 LLM을 활용한 애플리케이션을 쉽게 개발할 수 있도록 도와주는 프레임워크입니다.

### 핵심 구성 요소

1. **Models** - LLM, Chat Models, Embeddings
2. **Prompts** - 프롬프트 템플릿과 관리
3. **Chains** - 여러 컴포넌트를 연결
4. **Memory** - 대화 기록 관리
5. **Agents** - 도구를 사용하여 작업 수행

## 2. Chat Models(LLM) 기본 사용법

LangChain v1에서는 두 가지 방법으로 Chat Model을 초기화할 수 있습니다:

### 방법 1: `init_chat_model()`

`init_chat_model()`은 여러 제공자의 모델을 통일된 방식으로 초기화하는 가장 쉬운 방법입니다.

참고: https://docs.langchain.com/oss/python/langchain/models

In [3]:
# 방법 1: init_chat_model() 사용
from langchain.chat_models import init_chat_model

# 모델 이름만으로 간단하게 초기화
model = init_chat_model(
    "gpt-4o-mini",  # 또는 "gpt-4o", "claude-sonnet-4-5-20250929" 등
    temperature=0.7,  # 0.0 (결정적) ~ 1.0 (창의적)
    max_tokens=1000,  # 최대 출력 토큰 수
)

# 간단한 질문 실행
response = model.invoke("LangChain의 주요 장점 3가지를 알려주세요.")
print(response.content)

LangChain의 주요 장점은 다음과 같습니다:

1. **모듈화 및 확장성**: LangChain은 다양한 모듈을 갖추고 있어 사용자가 필요에 따라 특정 기능을 쉽게 추가하거나 제거할 수 있습니다. 이를 통해 복잡한 애플리케이션을 구축할 때 유연성과 확장성을 제공합니다.

2. **다양한 데이터 소스 통합**: LangChain은 여러 데이터 소스와 통합할 수 있는 기능을 제공하여, 사용자들이 구조화된 데이터, 비구조화된 데이터, API 호출 등을 통해 정보를 손쉽게 처리하고 사용할 수 있도록 합니다. 이를 통해 다양한 애플리케이션에서 활용할 수 있습니다.

3. **사용자 친화적인 API**: LangChain은 직관적인 API를 제공하여 개발자들이 쉽게 사용할 수 있도록 설계되었습니다. 이를 통해 비전문가도 자연어 처리 모델을 쉽게 활용하고, 복잡한 작업을 간단하게 수행할 수 있게 돕습니다.

이러한 장점 덕분에 LangChain은 자연어 처리 및 관련 애플리케이션 개발에 있어 유용한 도구로 자리잡고 있습니다.


### 주요 모델 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | 사용할 모델 이름 (예: "gpt-4o-mini") | 필수 |
| `temperature` | 출력의 무작위성 (0.0=결정적, 1.0=창의적) | 모델별 상이 |
| `max_tokens` | 최대 출력 토큰 수 | 모델별 상이 |
| `timeout` | API 요청 타임아웃 (초) | 없음 |
| `max_retries` | 실패 시 재시도 횟수 (지수 백오프 적용) | 6 |

### 방법 2: 직접 클래스 사용

`ChatOpenAI`를 직접 import하여 사용할 수도 있습니다. 제공자별 특화 기능이 필요한 경우 유용합니다.

In [9]:
# 방법 2: ChatOpenAI 직접 사용
from langchain_openai import ChatOpenAI

# ChatOpenAI 모델 초기화 (OpenAI 특화 기능 필요 시)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=1000,
)

response = llm.invoke("LangChain의 주요 장점 3가지를 알려주세요.")
print(response.content)

LangChain의 주요 장점은 다음과 같습니다:

1. **모듈화 및 유연성**: LangChain은 다양한 구성 요소로 구성되어 있어 사용자가 필요에 따라 특정 기능을 선택하고 조합할 수 있습니다. 이는 개발자들이 자신의 애플리케이션에 맞는 최적의 솔루션을 구축할 수 있게 해줍니다.

2. **고급 언어 모델 통합**: LangChain은 OpenAI, Hugging Face Transformers 등 여러 언어 모델과 쉽게 통합할 수 있는 기능을 제공합니다. 이를 통해 자연어 처리(NLP) 작업을 보다 간편하게 수행할 수 있으며, 다양한 모델의 성능을 활용할 수 있습니다.

3. **상황 인식 및 대화 관리**: LangChain은 대화형 애플리케이션을 구축할 때 필요한 상태 관리 및 대화의 흐름을 처리할 수 있는 기능을 제공합니다. 이는 사용자가 보다 자연스럽고 연속적인 대화를 나눌 수 있도록 도와줍니다.

이러한 장점들 덕분에 LangChain은 다양한 NLP 애플리케이션 및 대화형 시스템을 개발하는 데 유용한 도구로 자리잡고 있습니다.


In [10]:
response

AIMessage(content='LangChain의 주요 장점은 다음과 같습니다:\n\n1. **모듈화 및 유연성**: LangChain은 다양한 구성 요소로 구성되어 있어 사용자가 필요에 따라 특정 기능을 선택하고 조합할 수 있습니다. 이는 개발자들이 자신의 애플리케이션에 맞는 최적의 솔루션을 구축할 수 있게 해줍니다.\n\n2. **고급 언어 모델 통합**: LangChain은 OpenAI, Hugging Face Transformers 등 여러 언어 모델과 쉽게 통합할 수 있는 기능을 제공합니다. 이를 통해 자연어 처리(NLP) 작업을 보다 간편하게 수행할 수 있으며, 다양한 모델의 성능을 활용할 수 있습니다.\n\n3. **상황 인식 및 대화 관리**: LangChain은 대화형 애플리케이션을 구축할 때 필요한 상태 관리 및 대화의 흐름을 처리할 수 있는 기능을 제공합니다. 이는 사용자가 보다 자연스럽고 연속적인 대화를 나눌 수 있도록 도와줍니다.\n\n이러한 장점들 덕분에 LangChain은 다양한 NLP 애플리케이션 및 대화형 시스템을 개발하는 데 유용한 도구로 자리잡고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 255, 'prompt_tokens': 20, 'total_tokens': 275, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint'

## 3. 메시지(Messages) 이해하기

**메시지(Message)** 는 LangChain에서 모델과 대화하기 위한 기본 단위입니다.

### 3-1. 메시지의 구성 요소

1. **Role (역할)** - 메시지 타입 식별 (`system`, `user`)
2. **Content (내용)** - 실제 메시지 내용
3. **Metadata (메타데이터)** - 선택적 필드 (ID, 이름, 토큰 사용량 등)

In [11]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

response = model.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content="J'adore créer des applications." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 48, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DPeIvmpvncV4uWc7PDZFE24eiQxWm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d4692-79c4-7e31-901b-9c359f8de3cd-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 6, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [13]:
response.content

"J'adore créer des applications."

### 3-2. 텍스트 프롬프트 vs 메시지 프롬프트

| 구분 | 텍스트 프롬프트 | 메시지 프롬프트 |
|-----|--------------|---------------|
| 형식 | 문자열 | Message 객체 또는 딕셔너리 리스트 |
| 사용 시기 | 단순한 1회성 요청 | 대화 기록 관리, 멀티모달 콘텐츠 |
| 대화 기록 | 불가능 | 가능 |

참고: https://docs.langchain.com/oss/python/langchain/messages

**텍스트 프롬프트:**
```python
response = model.invoke("Write a haiku about spring")
```

**메시지 프롬프트:**
```python
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)
```

### 3-3. 메시지 타입 (Message Types)

LangChain은 4가지 주요 메시지 타입을 제공합니다:

**1) SystemMessage - 시스템 지시사항**
- 모델의 행동과 역할을 정의합니다. 대화의 맥락과 가이드라인을 설정합니다.

**2) HumanMessage - 사용자 입력**
- 사용자의 질문이나 요청을 나타냅니다. 텍스트, 이미지, 오디오 등 멀티모달 콘텐츠를 포함할 수 있습니다.

**3) AIMessage - AI 응답**
- 모델이 생성한 응답입니다. 텍스트 외에도 tool_calls, usage_metadata 등의 메타데이터를 포함합니다.

**4) ToolMessage - 도구 실행 결과**
- Tool 호출의 결과를 모델에 전달할 때 사용합니다.

In [14]:
# 텍스트 프롬프트 (단순한 문자열)
response = llm.invoke("AI 에이전트란 무엇인가요?")
print("=== 텍스트 프롬프트 ===")
print(response.content)
print(f"응답 타입: {type(response)}")
print()

# 메시지 프롬프트 (대화 기록 포함)
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("당신은 반말로 대답하는 AI 어시스턴트입니다."),
    HumanMessage("AI 에이전트란 무엇인가요?")
]
response = llm.invoke(messages)
print("=== 메시지 프롬프트 ===")
print(response.content)
print(f"응답 타입: {type(response)}")

=== 텍스트 프롬프트 ===
AI 에이전트란 특정 작업을 수행하거나 문제를 해결하기 위해 설계된 인공지능 시스템을 의미합니다. 이러한 에이전트는 환경과 상호작용하며, 주어진 목표를 달성하기 위해 자율적으로 행동할 수 있는 능력을 갖추고 있습니다. AI 에이전트는 다음과 같은 특징을 가집니다:

1. **자율성**: AI 에이전트는 인간의 개입 없이 스스로 결정을 내리고 행동할 수 있습니다.
  
2. **지식 기반**: AI 에이전트는 특정 도메인에 대한 지식을 가지고 있으며, 이를 활용하여 문제를 해결합니다.

3. **상호작용**: AI 에이전트는 환경이나 다른 에이전트와 상호작용하며 데이터를 수집하고 학습할 수 있습니다.

4. **목표 지향성**: AI 에이전트는 특정 목표를 가지고 이를 달성하기 위한 행동을 계획하고 실행합니다.

AI 에이전트의 예로는 챗봇, 자율주행차, 추천 시스템, 게임 AI 등이 있습니다. 이러한 시스템은 다양한 분야에서 활용되어 효율성을 높이고 사용자 경험을 개선하는 데 기여하고 있습니다.
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>

=== 메시지 프롬프트 ===
AI 에이전트는 인공지능을 기반으로 한 프로그램이나 시스템으로, 특정 작업을 수행하거나 문제를 해결하기 위해 설계된 거야. 이들은 주어진 환경에서 데이터나 정보를 수집하고, 분석하고, 결정을 내리거나 행동을 취할 수 있어. 예를 들어, 챗봇이나 자율주행차, 추천 시스템 등이 AI 에이전트의 예라고 할 수 있어.
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [15]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# SystemMessage 예제 - 상세한 페르소나 설정
system_msg = SystemMessage("""
당신은 10년 경력의 시니어 Python 개발자입니다.
항상 코드 예제를 제공하고 설명의 근거를 제시하세요.
간결하면서도 철저한 설명을 제공하세요.
""")

messages = [
    system_msg,
    HumanMessage(content="Python의 리스트 컴프리헨션에 대해 설명해주세요.")
]

response = llm.invoke(messages)
print(response.content)
print()

리스트 컴프리헨션(List Comprehension)은 Python에서 리스트를 간결하고 효율적으로 생성하는 방법입니다. 이 구문을 사용하면 기존 리스트 또는 다른 iterable 객체를 기반으로 새로운 리스트를 쉽게 만들 수 있습니다. 

### 기본 구문
리스트 컴프리헨션의 기본 구문은 다음과 같습니다:

```python
new_list = [expression for item in iterable if condition]
```

- `expression`: 새 리스트의 각 요소를 생성하는 표현식입니다.
- `item`: iterable에서 가져오는 요소입니다.
- `iterable`: 리스트, 튜플, 문자열 등 순회 가능한 객체입니다.
- `condition`: (선택 사항) 요소를 포함할지 결정하는 조건입니다.

### 예제 1: 기본 리스트 생성
리스트 컴프리헨션을 사용하여 0부터 9까지의 숫자의 제곱을 포함하는 리스트를 생성해 보겠습니다.

```python
squares = [x**2 for x in range(10)]
print(squares)
```

**출력:**
```
[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
```

이 코드는 `range(10)`으로부터 각 숫자 `x`를 가져와 `x**2`를 계산하여 새로운 리스트를 생성합니다.

### 예제 2: 조건을 사용하는 경우
리스트 컴프리헨션에서 조건을 추가하여 특정 조건을 만족하는 요소만 포함할 수 있습니다. 예를 들어, 0부터 9까지의 숫자 중 짝수의 제곱만 포함하는 리스트를 만들어 보겠습니다.

```python
even_squares = [x**2 for x in range(10) if x % 2 == 0]
print(even_squares)
```

**출력:**
```
[0, 4, 16, 36, 64]
```

여기서 `if x % 2 == 0` 조건을 사용하여 짝수만 선택하여 그 제곱을 계산합니다.

### 장점
- **가독성**: 리스트 컴프리헨션은 

In [16]:
# AIMessage를 수동으로 생성하여 대화 기록 구성
conversation = [
    SystemMessage("당신은 도움이 되는 AI 어시스턴트입니다."),
    HumanMessage("도와주실 수 있나요?"),
    AIMessage("물론입니다! 기꺼이 도와드리겠습니다."),  # 수동으로 추가한 AI 응답
    HumanMessage("2+2는 얼마인가요?")
]

response = llm.invoke(conversation)
print("=== 수동으로 구성한 대화 기록 ===")
print(f"최종 응답: {response.content}")
print("\n전체 대화:")
for i, msg in enumerate(conversation + [response], 1):
    print(f"{i}. [{msg.type}] {msg.content}")

=== 수동으로 구성한 대화 기록 ===
최종 응답: 2 + 2는 4입니다. 다른 질문이 있으신가요?

전체 대화:
1. [system] 당신은 도움이 되는 AI 어시스턴트입니다.
2. [human] 도와주실 수 있나요?
3. [ai] 물론입니다! 기꺼이 도와드리겠습니다.
4. [human] 2+2는 얼마인가요?
5. [ai] 2 + 2는 4입니다. 다른 질문이 있으신가요?


### 3-4. 메시지 메타데이터 설정

In [17]:
human_msg = HumanMessage(
    content="Hello!",
    name="alice",  # Optional: identify different users
    id="msg_123",  # Optional: unique identifier for tracing
)

In [18]:
human_msg

HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={}, name='alice', id='msg_123')

### 📖 과제 1: 시스템 프롬프트로 챗봇의 역할 정의하기
SystemMessage로 시스템 프롬프트(페르소나, 역할, 규칙 등 정의)를 작성하고 답변 받아보기


In [ ]:
# CODE HERE

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# SystemMessage 예제 - 상세한 페르소나 설정
system_msg = SystemMessage("""
당신은 10년 경력의 시니어 Python 개발자입니다.
항상 코드 예제를 제공하고 설명의 근거를 제시하세요.
간결하면서도 철저한 설명을 제공하세요.
""")

messages = [
    system_msg,
    HumanMessage(content="Python의 리스트 컴프리헨션에 대해 설명해주세요.")
]

response = llm.invoke(messages)
print(response.content)
print()

## 4. 대화 기록 관리

Chat Models는 기본적으로 stateless입니다. 대화 맥락을 유지하려면 메시지 리스트를 직접 관리해야 합니다.

### 대화 기록 관리 방법

1. 메시지 리스트에 HumanMessage 추가
2. 모델 호출
3. 반환된 AIMessage를 리스트에 추가
4. 다음 호출 시 전체 리스트 전달

In [19]:
# 대화 기록을 메시지 리스트로 관리
chat_history = []

# 첫 번째 대화
print("=== 대화 1 ===")
user_msg_1 = HumanMessage(content="제 이름은 민수입니다.")
chat_history.append(user_msg_1)

response_1 = llm.invoke(chat_history)
print(f"User: {user_msg_1.content}")
print(f"AI: {response_1.content}")

# AI 응답을 대화 기록에 추가
chat_history.append(response_1)

# 두 번째 대화
print("\n=== 대화 2 ===")
user_msg_2 = HumanMessage(content="제 이름이 뭐라고 했죠?")
chat_history.append(user_msg_2)

response_2 = llm.invoke(chat_history)
print(f"User: {user_msg_2.content}")
print(f"AI: {response_2.content}")
chat_history.append(response_2)

# 대화 기록 확인
print(f"\n현재 대화 기록 길이: {len(chat_history)} messages")
for i, msg in enumerate(chat_history, 1):
    print(f"{i}. [{msg.type}] {msg.content}")

=== 대화 1 ===
User: 제 이름은 민수입니다.
AI: 안녕하세요, 민수님! 어떻게 도와드릴까요?

=== 대화 2 ===
User: 제 이름이 뭐라고 했죠?
AI: 민수님이라고 하셨습니다! 다른 질문이나 도움이 필요하시면 말씀해 주세요.

현재 대화 기록 길이: 4 messages
1. [human] 제 이름은 민수입니다.
2. [ai] 안녕하세요, 민수님! 어떻게 도와드릴까요?
3. [human] 제 이름이 뭐라고 했죠?
4. [ai] 민수님이라고 하셨습니다! 다른 질문이나 도움이 필요하시면 말씀해 주세요.


In [20]:
# SystemMessage로 역할 설정 + 대화 기록 관리
conversation = [
    SystemMessage(content="당신은 Python 튜터입니다. 초보자에게 친절하게 설명하세요."),
    HumanMessage(content="변수란 무엇인가요?"),
]

response = llm.invoke(conversation)
print("=== 역할이 설정된 대화 ===")
print(f"AI: {response.content}\n")

# 대화 이어가기
conversation.append(response)
conversation.append(HumanMessage(content="예제를 보여주세요."))

response = llm.invoke(conversation)
print("=== 이어진 대화 ===")
print(f"AI: {response.content}")

=== 역할이 설정된 대화 ===
AI: 변수는 프로그래밍에서 데이터를 저장하는 "상자"와 같은 역할을 합니다. 쉽게 말해서, 변수를 사용하면 특정 값을 기억해 놓고, 나중에 그 값을 다시 사용할 수 있습니다.

예를 들어, 당신이 숫자 10을 기억하고 싶다면, 변수를 사용하여 그 숫자를 저장할 수 있습니다. Python에서 변수를 만들려면 다음과 같이 할 수 있습니다:

```python
my_number = 10
```

여기서 `my_number`는 변수의 이름이고, `10`은 그 변수에 저장된 값입니다. 이제 `my_number`라고 쓰면 언제든지 10이라는 값을 사용할 수 있습니다.

변수의 장점은 다음과 같습니다:

1. **재사용성**: 한 번 저장한 값을 여러 번 사용할 수 있습니다.
2. **가독성**: 코드에서 의미 있는 이름을 사용하여 어떤 데이터를 저장하고 있는지 쉽게 이해할 수 있습니다.
3. **유연성**: 필요에 따라 값을 언제든지 바꿀 수 있습니다.

예를 들어, `my_number`에 다른 값을 할당할 수도 있습니다:

```python
my_number = 20
```

이제 `my_number`는 더 이상 10이 아니라 20이 됩니다.

변수는 다양한 종류의 데이터를 저장할 수 있습니다. 예를 들어, 숫자, 문자, 리스트 등 여러 가지 형태로 데이터를 저장할 수 있습니다. 

이처럼 변수를 사용하면 더 효율적이고 유연한 프로그램을 만들 수 있습니다!

=== 이어진 대화 ===
AI: 물론입니다! 변수를 사용하는 간단한 예제를 보여드릴게요. 이 예제에서는 두 개의 숫자를 더하고 결과를 출력하는 프로그램을 만들어 보겠습니다.

```python
# 두 개의 숫자를 변수에 저장합니다
number1 = 5
number2 = 10

# 두 숫자를 더합니다
result = number1 + number2

# 결과를 출력합니다
print("두 숫자의 합은:", result)
```

이 코드를 단계별로 설명해드릴게요:

1. **변수 

### 📖 과제 2: 멀티턴
1. SystemMessage로 "당신은 여행 가이드입니다"로 역할 설정
2. 사용자가 "일본 여행을 가려고 해요" 질문
3. AI 응답 받기
4. 이어서 "어디를 추천하나요?" 질문
5. 전체 대화 기록 출력

In [ ]:
# CODE HERE

In [21]:
# SystemMessage로 역할 설정 + 대화 기록 관리
conversation = [
    SystemMessage(content="당신은 여행 가이드입니다. 여행 장소에 대한 여행 계획이나 참고할만한 정보를 제공하세요."),
    HumanMessage(content="일본 여행을 가려고 해요."),
]

response = llm.invoke(conversation)
print("=== 역할이 설정된 대화 ===")
print(f"AI: {response.content}\n")

# 대화 이어가기
conversation.append(response)
conversation.append(HumanMessage(content="추천 여행 코스를 알려주세요."))

response = llm.invoke(conversation)
print("=== 이어진 대화 ===")
print(f"AI: {response.content}")

=== 역할이 설정된 대화 ===
AI: 일본은 다양한 문화와 아름다운 경관, 맛있는 음식이 가득한 멋진 여행지입니다. 아래는 일본 여행을 위한 추천 일정과 장소입니다.

### 1. 도쿄 (Tokyo)
- **추천 일정**: 3-4일
- **주요 관광지**:
  - **아사쿠사**: 센소지 사원과 나카미세 거리에서 전통적인 일본 문화를 체험해 보세요.
  - **도쿄 타워**: 멋진 전망을 즐기며 도쿄의 전경을 감상할 수 있습니다.
  - **하라주쿠**: 트렌디한 패션과 독특한 카페 문화가 있는 지역입니다.
  - **신주쿠**: 다양한 음식과 나이트 라이프를 즐길 수 있는 곳입니다.

### 2. 교토 (Kyoto)
- **추천 일정**: 2-3일
- **주요 관광지**:
  - **금각사 (킨카쿠지)**: 아름다운 금빛 사원과 정원이 인상적입니다.
  - **기온**: 전통적인 게이샤 문화를 경험할 수 있는 지역입니다.
  - **아라시야마**: 대나무 숲과 후시미 이나리 신사 방문을 추천합니다.

### 3. 오사카 (Osaka)
- **추천 일정**: 1-2일
- **주요 관광지**:
  - **오사카 성**: 역사적인 성과 아름다운 정원을 둘러보세요.
  - **도톤보리**: 유명한 길거리 음식을 맛볼 수 있는 지역으로, 특히 타코야키와 오코노미야키가 유명합니다.

### 4. 히로시마 (Hiroshima)
- **추천 일정**: 1일
- **주요 관광지**:
  - **평화 기념 공원**: 히로시마 원폭의 역사를 배우고 기념하는 장소입니다.
  - **미야지마 섬**: 이츠쿠시마 신사의 떠 있는 토리이 문이 유명합니다.

### 5. 홋카이도 (Hokkaido) - 겨울 여행 추천
- **추천 일정**: 3-5일
- **주요 관광지**:
  - **삿포로**: 삿포로 맥주 박물관과 스노우 페스티벌을 즐길 수 있습니다.
  - **니세코**: 스키와 스노보드를 즐길 수 있는 유명한 스키 리조트입니다.

### 여행 팁
- **교통**: 일본의 대중

In [22]:
conversation.append(HumanMessage(content="난 도쿄만 갈래"))

response = llm.invoke(conversation)
print("=== 이어진 대화 ===")
print(f"AI: {response.content}")

=== 이어진 대화 ===
AI: 도쿄에서의 여행은 다양한 문화와 즐길 거리로 가득합니다. 아래는 도쿄에서 4일 동안 즐길 수 있는 추천 여행 코스입니다.

### 1일차: 전통과 현대의 만남
- **아사쿠사**  
  - 아침: 센소지 사원 방문. 일본에서 가장 유명한 사원 중 하나이며, 주변의 나카미세 거리에서 기념품과 전통 간식을 즐기세요.
  
- **우에노 공원**  
  - 점심: 공원 내의 음식점이나 벤토를 구매하여 피크닉을 즐기세요. 우에노 동물원과 박물관도 방문할 수 있습니다.

- **시부야**  
  - 오후: 시부야 스크램블 교차로를 경험하고, 하치코 동상을 찾아보세요. 인근 쇼핑 지구를 탐방하며 다양한 브랜드와 카페를 즐겨보세요.

- **하라주쿠**  
  - 저녁: 다케시타 거리에서 독특한 패션과 기념품 쇼핑을 즐기세요. 근처의 오모테산도에서 고급 상점도 구경할 수 있습니다.

### 2일차: 문화 체험
- **교외 탐방**  
  - 아침: 메이지 신궁 방문. 아름다운 숲 속에서 조용한 산책을 즐길 수 있습니다.
  
- **도쿄 미드타운**  
  - 점심: 미드타운 내의 레스토랑에서 다양한 요리를 맛보세요.

- **신주쿠**  
  - 오후: 도쿄 도청사 전망대에서 도쿄 전경을 감상하세요. 무료로 이용할 수 있습니다.
  - 저녁: 신주쿠의 이자카야에서 저녁 식사를 즐기며 일본의 나이트 라이프를 경험해보세요.

### 3일차: 현대적인 매력
- **오다이바**  
  - 아침: 유리카모메를 타고 오다이바로 가세요. 다이버시티 도쿄 플라자에서 쇼핑과 다양한 체험을 즐기세요.
  
- **팀랩 보더리스**  
  - 오후: 디지털 아트 뮤지엄인 팀랩 보더리스를 방문하여 환상적인 아트 전시를 경험하세요.

- **롯폰기**  
  - 저녁: 롯폰기 힐즈에서 멋진 야경을 감상하고, 인근의 레스토랑에서 저녁을 즐기세요.

### 4일차: 쇼핑과 휴식
- **긴자**  
  - 아침: 고급 브랜드 매장이 즐비한 긴자에서 쇼핑을 즐기세요.

## 5. 스트리밍 (Streaming)

### 5-1. AIMessageChunk

- `invoke()`는 하나의 `AIMessage`를 반환
- `stream()`은 여러 개의 `AIMessageChunk`를 반환
- 각 청크는 출력 텍스트의 일부를 포함
- 더하기 연산으로 전체 메시지를 구성 가능

In [25]:
# 방법 1: 텍스트만 스트리밍
for chunk in llm.stream("Python의 장점을 간단히 설명해주세요."):
    # print(chunk)
    print(chunk.text, end="", flush=True)
print("\n")

Python의 장점은 다음과 같습니다:

1. **간결하고 읽기 쉬운 문법**: Python은 문법이 간단하고 직관적이어서 초보자도 쉽게 배울 수 있습니다. 코드가 읽기 쉬워 유지보수와 협업에 유리합니다.

2. **풍부한 라이브러리**: 데이터 분석, 웹 개발, 머신러닝 등 다양한 분야에 사용할 수 있는 방대한 라이브러리와 프레임워크가 제공되어 개발 속도를 높일 수 있습니다.

3. **다양한 응용 분야**: Python은 웹 개발(Django, Flask), 데이터 과학(Pandas, NumPy), 인공지능(TensorFlow, PyTorch), 자동화 스크립트 등 다양한 분야에서 사용됩니다.

4. **활발한 커뮤니티**: Python은 큰 사용자 커뮤니티를 가지고 있어 문제 해결이나 정보 공유가 용이합니다.

5. **플랫폼 독립성**: Python은 다양한 운영체제에서 실행될 수 있어, 코드의 이식성이 뛰어납니다.

6. **객체지향 및 함수형 프로그래밍 지원**: Python은 객체지향 프로그래밍(OOP)과 함수형 프로그래밍을 모두 지원하여 유연한 코딩 스타일을 제공합니다.

이러한 장점들 덕분에 Python은 많은 개발자와 기업에서 선호되는 프로그래밍 언어입니다.



### 5-2. AIMessageChunk 누적하기

스트리밍 중에 청크를 누적하여 전체 AIMessage를 구성할 수 있습니다. 누적된 메시지는 `invoke()`로 받은 메시지와 동일하게 사용할 수 있습니다.

In [26]:
# 방법 2: AIMessageChunk 누적하여 전체 메시지 구성
print("=== 청크 누적 방식 ===")
full_message = None

for chunk in llm.stream("파이썬이란 무엇인가요?"):
    # 청크를 더하기 연산으로 누적
    full_message = chunk if full_message is None else full_message + chunk
    print(chunk.text, end="", flush=True)

print(f"\n\n최종 메시지 타입: {type(full_message)}")
print(f"최종 메시지 content: {full_message.content}")

=== 청크 누적 방식 ===
파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 코드의 가독성이 높고, 문법이 간결하여 배우기 쉽고, 다양한 용도로 사용할 수 있는 언어입니다. 

주요 특징은 다음과 같습니다:

1. **가독성**: 파이썬은 코드가 명확하고 간결하여 다른 사람들이 이해하기 쉽게 작성할 수 있습니다.
2. **다양한 라이브러리**: 데이터 과학, 웹 개발, 인공지능, 자동화 등 다양한 분야에 사용할 수 있는 많은 라이브러리와 프레임워크가 존재합니다. 예를 들어, NumPy, Pandas, TensorFlow, Django 등이 있습니다.
3. **플랫폼 독립성**: 파이썬은 다양한 운영체제에서 실행할 수 있습니다. Windows, macOS, Linux 등에서 사용 가능합니다.
4. **인터프리터 언어**: 파이썬은 인터프리터 언어로, 코드를 한 줄씩 실행할 수 있어 디버깅이 쉬운 장점이 있습니다.
5. **객체 지향 프로그래밍**: 파이썬은 객체 지향 프로그래밍을 지원하여, 코드의 재사용성과 모듈화를 촉진합니다.

이러한 특징들 덕분에 파이썬은 프로그래머, 데이터 과학자, 웹 개발자 등 다양한 분야에서 널리 사용되고 있습니다.

최종 메시지 타입: <class 'langchain_core.messages.ai.AIMessageChunk'>
최종 메시지 content: 파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 코드의 가독성이 높고, 문법이 간결하여 배우기 쉽고, 다양한 용도로 사용할 수 있는 언어입니다. 

주요 특징은 다음과 같습니다:

1. **가독성**: 파이썬은 코드가 명확하고 간결하여 다른 사람들이 이해하기 쉽게 작성할 수 있습니다.
2. **다양한 라이브러리**: 데이터 과학, 웹 개발, 인공지능, 자동화 등 다양한 분야에 사용할 수 있는 많

### 📖 과제 3: 스트리밍으로 긴 설명 받기
"인공지능의 역사를 상세히 설명해주세요"를 stream()으로 처리하고,
청크가 올 때마다 화면에 출력하세요.

In [27]:
# CODE HERE
# 방법 1: 텍스트만 스트리밍
for chunk in llm.stream("인공지능의 역사를 상세히 설명해주세요."):
    # print(chunk)
    print(chunk.text, end="", flush=True)
print("\n")

인공지능(AI)의 역사는 20세기 중반부터 시작되어 현재까지 이어져오고 있으며, 여러 중요한 발전과 이정표가 포함되어 있습니다. 아래에 인공지능의 역사적 발전을 상세히 설명하겠습니다.

### 1. 초기 개념 (1940년대~1950년대)
- **1943년**: 워렌 매카시와 월터 피츠가 뉴런의 작동을 모델링하여 인공지능의 기초가 되는 신경망 이론을 제안합니다.
- **1950년**: 앨런 튜링이 "Computing Machinery and Intelligence"라는 논문을 발표하며 튜링 테스트를 제안합니다. 이는 기계가 인간처럼 사고할 수 있는지를 평가하는 방법입니다.

### 2. 인공지능의 출생 (1956년)
- **1956년**: 다트머스 회의에서 "인공지능"이라는 용어가 처음 사용됩니다. 이는 존 매카시, 마빈 민스키, 나단 미어비어, 클로드 섀넌 등 여러 연구자들이 모여 AI의 발전을 논의한 자리입니다. 이 회의는 AI 연구의 출발점으로 간주됩니다.

### 3. 초기 발전 (1950년대~1960년대)
- **1950년대 후반**: 초기 AI 프로그램들이 개발됩니다. 예를 들어, 뉴엘과 샘슨이 개발한 "Logic Theorist"와 "General Problem Solver"는 문제 해결을 위한 알고리즘을 제시했습니다.
- **1966년**: 조셉 웨이젠바움이 개발한 ELIZA는 자연어 처리를 이용한 초기 챗봇으로, 사용자의 입력에 대한 반응을 통해 대화를 시뮬레이션했습니다.

### 4. 첫 번째 AI 겨울 (1970년대)
- **1970년대**: AI에 대한 과도한 기대와 실제 성과의 괴리로 인해 자금 지원이 감소하고 연구가 둔화되는 "AI 겨울"이 발생합니다. 이 시기에 많은 프로젝트가 중단되거나 실패합니다.

### 5. 전문가 시스템의 발전 (1980년대)
- **1980년대**: 전문가 시스템이 인기를 끌며 AI 연구가 다시 활성화됩니다. MYCIN, DENDRAL 등 특정 분야에서 전문가의 지식을 모방하는 시스템이 개발됩니다.
- **1

---

### 참고 자료

- [LangChain Models 공식 문서](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain Messages 공식 문서](https://docs.langchain.com/oss/python/langchain/messages)
- [Chat Models 통합 가이드](https://docs.langchain.com/oss/python/integrations/chat)